## Training

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler

MODEL_DATA_DIR = Path('model_data')

N_VALUES = [5, 10, 15, 20]

FEATURES = [
    'elo_diff',
    'form_diff',
    'comp_form_diff',
    'home_elo',
    'away_elo',
    'home_form',
    'away_form',
    'home_comp_form',
    'away_comp_form',
]

def load_and_engineer(path):
    df = pd.read_csv(path)
    df['elo_diff']       = df['home_elo']       - df['away_elo']
    df['form_diff']      = df['home_form']       - df['away_form']
    df['comp_form_diff'] = df['home_comp_form']  - df['away_comp_form']
    return df

print('Imports ready.')

Imports ready.


### Logistic Regression

In [2]:
lr_results = []

for n in N_VALUES:
    train = load_and_engineer(MODEL_DATA_DIR / f'training_data_{n}.csv')
    val   = load_and_engineer(MODEL_DATA_DIR / f'validation_data_{n}.csv')

    X_train, y_train = train[FEATURES], train['result']
    X_val,   y_val   = val[FEATURES],   val['result']

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    train_acc   = accuracy_score(y_train, model.predict(X_train))
    val_acc     = accuracy_score(y_val,   model.predict(X_val))
    val_logloss = log_loss(y_val, model.predict_proba(X_val))

    lr_results.append({
        'N':           n,
        'train_acc':   round(train_acc,   4),
        'val_acc':     round(val_acc,     4),
        'val_logloss': round(val_logloss, 4),
    })
    print(f'N={n:2d} | train acc: {train_acc:.3f} | val acc: {val_acc:.3f} | val log-loss: {val_logloss:.3f}')

lr_results_df = pd.DataFrame(lr_results)
print()
lr_results_df

N= 5 | train acc: 0.549 | val acc: 0.531 | val log-loss: 1.068
N=10 | train acc: 0.568 | val acc: 0.547 | val log-loss: 1.057
N=15 | train acc: 0.552 | val acc: 0.531 | val log-loss: 1.095
N=20 | train acc: 0.557 | val acc: 0.531 | val log-loss: 1.081



,N,train_acc,val_acc,val_logloss
0,5,0.5495,0.5312,1.0681
1,10,0.5677,0.5469,1.0569
2,15,0.5521,0.5312,1.0945
3,20,0.5573,0.5312,1.0806


### Gradient Boosted Trees

In [3]:
from sklearn.ensemble import GradientBoostingClassifier

gbt_results = []

for n in N_VALUES:
    train = load_and_engineer(MODEL_DATA_DIR / f'training_data_{n}.csv')
    val   = load_and_engineer(MODEL_DATA_DIR / f'validation_data_{n}.csv')

    X_train, y_train = train[FEATURES].values, train['result'].values
    X_val,   y_val   = val[FEATURES].values,   val['result'].values

    model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42, learning_rate=0.1)
    model.fit(X_train, y_train) # type: ignore

    train_acc   = accuracy_score(y_train, model.predict(X_train)) # type: ignore
    val_acc     = accuracy_score(y_val,   model.predict(X_val)) # type: ignore
    val_logloss = log_loss(y_val, model.predict_proba(X_val)) # type: ignore

    gbt_results.append({
        'N':           n,
        'train_acc':   round(train_acc,   4),
        'val_acc':     round(val_acc,     4),
        'val_logloss': round(val_logloss, 4),
    })
    print(f'N={n:2d} | train acc: {train_acc:.3f} | val acc: {val_acc:.3f} | val log-loss: {val_logloss:.3f}')

gbt_results_df = pd.DataFrame(gbt_results)
print()
gbt_results_df

N= 5 | train acc: 0.966 | val acc: 0.422 | val log-loss: 1.239
N=10 | train acc: 0.966 | val acc: 0.469 | val log-loss: 1.129
N=15 | train acc: 0.971 | val acc: 0.484 | val log-loss: 1.341
N=20 | train acc: 0.958 | val acc: 0.453 | val log-loss: 1.229



,N,train_acc,val_acc,val_logloss
0,5,0.9661,0.4219,1.2385
1,10,0.9661,0.4688,1.1294
2,15,0.9714,0.4844,1.3406
3,20,0.9583,0.4531,1.2294


### XGBoost

In [5]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

xgb_results = []

# Encode labels to 0,1,2 as XGBoost requires non-negative integer classes
le = LabelEncoder()

for n in N_VALUES:
    train = load_and_engineer(MODEL_DATA_DIR / f'training_data_{n}.csv')
    val   = load_and_engineer(MODEL_DATA_DIR / f'validation_data_{n}.csv')

    X_train, y_train = train[FEATURES].values, le.fit_transform(train['result'].values) # type: ignore
    X_val,   y_val   = val[FEATURES].values,   le.transform(val['result'].values) # type: ignore

    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        reg_lambda=2.0,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0,
    )
    model.fit(X_train, y_train)

    train_acc   = accuracy_score(y_train, model.predict(X_train))
    val_acc     = accuracy_score(y_val,   model.predict(X_val))
    val_logloss = log_loss(y_val, model.predict_proba(X_val))

    xgb_results.append({
        'N':           n,
        'train_acc':   round(train_acc,   4),
        'val_acc':     round(val_acc,     4),
        'val_logloss': round(val_logloss, 4),
    })
    print(f'N={n:2d} | train acc: {train_acc:.3f} | val acc: {val_acc:.3f} | val log-loss: {val_logloss:.3f}')

xgb_results_df = pd.DataFrame(xgb_results)
print()
xgb_results_df

N= 5 | train acc: 0.914 | val acc: 0.422 | val log-loss: 1.132
N=10 | train acc: 0.930 | val acc: 0.531 | val log-loss: 1.078
N=15 | train acc: 0.919 | val acc: 0.516 | val log-loss: 1.224
N=20 | train acc: 0.901 | val acc: 0.469 | val log-loss: 1.169



,N,train_acc,val_acc,val_logloss
0,5,0.9141,0.4219,1.1321
1,10,0.9297,0.5312,1.0776
2,15,0.9193,0.5156,1.2236
3,20,0.9010,0.4688,1.1692
